# Reproduce SMITH Figure 6c-d

This notebook regenerates manuscript-matched data panels from real H5AD inputs; it does not read the packaged reference-output tables. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/agent_section/05_SMITH_Agent_Evaluation_source.ipynb).

## Configure real inputs and fresh outputs

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'agent'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 1))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Verify input files

In [ ]:
inputs = ['agent/liver_merfish/adata_healthy_nucseq.h5ad', 'agent/liver_merfish/adata_healthy_merfish.h5ad', 'agent/references/PSC011_C1_visium.h5ad', 'agent/references/WSSS_F_IMMsp9838712_visium.h5ad']
rows = []
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    rows.append({"file": relative, "bytes": path.stat().st_size, "sha256": sha256_file(path)})
display(pd.DataFrame(rows))


## Run the workflow for Figure 6c-d

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/agent/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--reference', 'references/PSC011_C1_visium.h5ad', '--reference', 'references/WSSS_F_IMMsp9838712_visium.h5ad', '--panel-sizes', '32,64,128', '--training-seeds', '1,2', '--max-cells', '3000'] + ["--force"]
display_command = ["python", 'reproducibility/workflows/agent/run_tutorial.py', "--data-root", "data/tutorials", "--output-dir", "outputs/tutorials/agent", "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--reference', 'references/PSC011_C1_visium.h5ad', '--reference', 'references/WSSS_F_IMMsp9838712_visium.h5ad', '--panel-sizes', '32,64,128', '--training-seeds', '1,2', '--max-cells', '3000'] + ["--force"]
print(" ".join(display_command))
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
print("Generated", manifest["manuscript_figure"], "data from fresh workflow outputs.")


## Inspect newly generated figure data

In [ ]:
for relative in ['figure_data/figure6_c_cell_type_accuracy.tsv', 'figure_data/figure6_d_merfish_expression.tsv']:
    path = CASE_OUTPUT / relative
    print(relative)
    display(pd.read_csv(path, sep="\t").head(20))


## Draw separate manuscript panels

Every panel below has its own canvas and manuscript-matched aspect ratio. The quick hosted run uses only the methods/repeats executed above; use the full command to regenerate the complete multi-method comparison.

In [ ]:
figure_dir = CASE_OUTPUT / "figures"
plot_command = [sys.executable, str(ROOT / 'reproducibility/workflows/agent/plot_figure6.py'), '--accuracy', str(CASE_OUTPUT / 'figure_data/figure6_c_cell_type_accuracy.tsv'), '--expression', str(CASE_OUTPUT / 'figure_data/figure6_d_merfish_expression.tsv'), "--output-dir", str(figure_dir)]
subprocess.run(plot_command, cwd=ROOT, check=True)
for heading, relative, width in [('Figure 6c - MERFISH cell-type accuracy', 'figures/figure6_c.png', 430), ('Figure 6d - MERFISH expression support', 'figures/figure6_d.png', 430)]:
    display(Markdown(f"### {heading}"))
    display(Image(filename=str(CASE_OUTPUT / relative), width=width))
print("Each panel is also exported independently as editable PDF/SVG and 600-dpi TIFF under", figure_dir)


## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--panel-sizes 32,64,128 --training-seeds 1,2,3,4,5 --epochs 200 (omit --reference to use all five manifest-listed defaults)
```

This hosted run uses two real spatial references and two training seeds. The manuscript Figure 6c-d command uses five retrieved liver references and five training seeds. Figure 6e-j requires external probe-design backends and the validation-guided HPO run; this notebook does not fabricate those panels.